# 05 Trip Durations

This notebook builds a simple city/year trip-duration summary for dashboard integration tests.

Output export: `trip_durations_city_year.csv`

In [1]:
import sys
import pandas as pd
import plotly.express as px
from ipywidgets import interact, IntSlider, Dropdown

sys.path.insert(0, "..")
from utils import load_app_ready, export_df

2026-03-29 00:26:38.763 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.764 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.765 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.765 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.766 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.766 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.767 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
df = load_app_ready()
df.shape

2026-03-29 00:26:38.772 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.773 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 00:26:38.775 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.776 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 00:26:38.777 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-29 00:26:38.777 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 00:26:38.777 WARNING streamlit.runtime.cachi

(15907082, 22)

In [3]:
required = ["city_name", "year", "duration_seconds"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns for duration analysis: {missing}")

duration_df = df.dropna(subset=["city_name", "year", "duration_seconds"]).copy()
duration_df["duration_minutes"] = duration_df["duration_seconds"] / 60

summary = (
    duration_df.groupby(["city_name", "year"], as_index=False)
    .agg(
        trips=("trip_id", "count"),
        mean_minutes=("duration_minutes", "mean"),
        median_minutes=("duration_minutes", "median"),
        p90_minutes=("duration_minutes", lambda s: s.quantile(0.9)),
    )
    .sort_values(["city_name", "year"])
    .reset_index(drop=True)
)

for c in ["mean_minutes", "median_minutes", "p90_minutes"]:
    summary[c] = summary[c].round(2)

summary.head()

,city_name,year,trips,mean_minutes,median_minutes,p90_minutes
0,Bergen,2018,103744,14.88,8.42,21.85
1,Bergen,2019,940004,11.13,7.80,19.88
2,Bergen,2020,1074590,10.67,7.93,18.93
3,Bergen,2021,656883,9.82,7.92,16.95
4,Bergen,2022,535316,9.87,7.95,16.62


## Interactive histogram (same logic as Analysis page)

Use the controls below to slice by city and adjust histogram cutoff in minutes.

In [ ]:
def show_duration_histogram(city_name="All Cities", max_minutes=60):
    scoped = duration_df if city_name == "All Cities" else duration_df[duration_df["city_name"] == city_name]
    minutes = scoped["duration_seconds"].dropna() / 60
    minutes = minutes[minutes <= max_minutes]

    fig = px.histogram(
        minutes,
        nbins=60,
        title=f"Trip Duration Distribution (<= {int(max_minutes)} min)" + ("" if city_name == "All Cities" else f" - {city_name}"),
        labels={"value": "Duration (minutes)", "count": "Count"},
        color_discrete_sequence=["#4ECDC4"],
    )
    fig.update_layout(showlegend=False, plot_bgcolor="white", margin=dict(t=40), height=460)
    fig.show()

city_options = ["All Cities"] + sorted(duration_df["city_name"].dropna().unique().tolist())
interact(
    show_duration_histogram,
    city_name=Dropdown(options=city_options, value="All Cities", description="City"),
    max_minutes=IntSlider(value=60, min=10, max=120, step=5, description="Cutoff"),
)

interactive(children=(Dropdown(description='City', options=('All Cities', 'Bergen', 'Oslo', 'Trondheim'), valu…

<function __main__.show_duration_histogram(city_name='All Cities', max_minutes=60)>

In [5]:
export_name = "trip_durations_city_year"
export_df(export_name, summary)
print(f"Exported: {export_name}.csv")

[bridge] Exported DataFrame → c:\Users\matia\Desktop\Projects\bysykkel\Urban-Cycling\02_data\gold\notebook_exports\trip_durations_city_year.csv
Exported: trip_durations_city_year.csv


## Notes

- Export path: `02_data/gold/notebook_exports/trip_durations_city_year.csv`
- This file is consumed by the Streamlit page `04_trip_durations.py`.